# VNA Stage B3 — iCaRL, measure only (Notebook 2)
Enable Internet, T4×2 and secret `github_token`; attach CIFAR-100 and optional
prior Stage-B archives. The immutable pin must include this implementation.
GPU0 runs PyCIL's own pipeline with one GPU; GPU1 is reserved for B2 then B4.
Only the three tiny B3 CPU tests run. Smoke uses two tasks × two epochs on real
data and includes the full measurement cost. The budget stops an over-budget
run. Hourly archives preserve logs; task resume is unavailable, so a crash
restarts B3 training. Completed runs are reused. No correction is applied.
Reproduction compares mean NME top-1 across ten tasks with 64.42 ±1.5pp;
last-task reference ≈49.5 is recorded separately. Smoke/fake data cannot pass.


In [ ]:
REPO_COMMIT = "d5e4e9ea55555503a270ca7bea868dd8026d7d1b"
GPU_HOURS_BUDGET = 8
ALLOW_OVER_BUDGET = False
MU_REL_DEFAULT = 1e-2
DISK_CAP_BYTES = 5_000_000_000


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_stage_b_nb2")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_vna_stage_b3.py"], cwd=REPO,
               env={**test_env, "CUDA_VISIBLE_DEVICES": ""}, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "diagnostics/signal_curve.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO), str(GROMO_SRC), str(REFERENCE)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")


In [ ]:
from experiments.stage_b5 import restore_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs" / "stageB"
STAGE_B.mkdir(parents=True, exist_ok=True)
restore_archives(Path("/kaggle/input"), STAGE_B)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python"}
archive_lock = threading.Lock()
archive_stop = threading.Event()
archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/stageB",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def command(out, *, smoke=False):
    cmd = [sys.executable, "-u", "-m", "experiments.stage_b3", "--out", str(out),
           "--device", "cuda:0", "--workers", "4", "--data-dir", str(DATA_ROOT),
           "--mu-rel-default", str(MU_REL_DEFAULT)]
    return cmd + (["--smoke"] if smoke else [])
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()


In [ ]:
# Real-data smoke: 2 tasks × 2 epochs, full ranks/sites/128-sample diagnostics.
SMOKE = RUN_ROOT / "smoke" / "outputs" / "stageB" / "b3"
COMPLETED = (STAGE_B / "b3" / "summary.json").exists()
try:
    if not COMPLETED:
        started = time.monotonic()
        subprocess.run(command(SMOKE, smoke=True), cwd=REPO, env=ENV, check=True)
        smoke_seconds = time.monotonic()-started
        summary = json.loads((SMOKE / "summary.json").read_text())
        # 200 initial + 9*170 incremental epochs; 4 points per task.
        training = summary["training_seconds"] * 1730 / summary["epochs_run"]
        point = max(p["seconds"] for p in summary["measurement_point_seconds"])
        projected = (training + 40*point)/3600
        total = projected + smoke_seconds/3600
        projection = {"projected_gpu_hours": projected, "smoke_gpu_hours": smoke_seconds/3600,
                      "total_gpu_hours": total, "full_epochs": 1730, "full_measurement_points": 40}
        (STAGE_B / "b3_cost_projection.json").write_text(json.dumps(projection, indent=2))
        print(projection)
        archive_outputs()
        if total > GPU_HOURS_BUDGET and not ALLOW_OVER_BUDGET:
            raise RuntimeError("Projected B3 GPU-hours exceed budget; review before ALLOW_OVER_BUDGET")
    else:
        print("Reusing completed B3 run")
except Exception:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs(); raise


In [ ]:
# GPU1 PLACEHOLDER — B2 then B4 will be added later.
# No GPU1 training job is launched by this version.


In [ ]:
try:
    if not COMPLETED:
        with (RUN_ROOT / "b3_gpu0.log").open("a", buffering=1) as log:
            subprocess.run(command(STAGE_B / "b3"), cwd=REPO, env=ENV, check=True,
                           stdout=log, stderr=subprocess.STDOUT)
finally:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs()
if archive_errors: raise RuntimeError(archive_errors)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
B3 = STAGE_B / "b3"
print("REPRODUCTION GATE", (B3 / "reproduction_gate.json").read_text())
print("GATE G", (B3 / "gate.json").read_text())
gate = json.loads((B3 / "gate.json").read_text())
if not gate["measurements_valid"]:
    print("*** INVALID MEASUREMENTS: reproduction did not pass ***")
rows = pd.read_csv(B3 / "measurements.csv")
for (rank, mu), group in rows.groupby(["rank", "mu_rel"]):
    points = group.groupby(["task", "epoch"]).rho_joint.mean()
    x = [task + epoch/(200 if task == 1 else 170) for task, epoch in points.index]
    plt.plot(x, points.values, marker="o", label=f"r={rank}, mu={mu}")
plt.xlabel("task + epoch / task epochs"); plt.ylabel("rho_joint")
plt.title("B3 rho vs task/epoch"); plt.legend()
plt.savefig(B3 / "rho_vs_task_epoch.png", bbox_inches="tight"); plt.show()
display(rows[["task", "epoch", "site", "rank", "rho_joint", "rho_exemplar", "rho_new"]])
archive_outputs()
